In [ ]:
import pandas as pd

# Load orders data
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')

# Convert timestamps
ts_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in ts_cols:
    orders[col] = pd.to_datetime(orders[col])

# Engineer time-difference columns (great features later)
orders['purchased_delivered'] = (
    orders.order_delivered_customer_date - orders.order_purchase_timestamp
).dt.days
orders['delivered_estimated'] = (
    orders.order_estimated_delivery_date - orders.order_delivered_customer_date
).dt.days

# Keep ONLY delivered orders
orders = orders[orders['order_status'] == 'delivered']
orders = orders.dropna(subset=['order_delivered_customer_date'])
print('Delivered orders:', orders.shape[0])

Delivered orders: 96470


In [4]:
# Load payments data
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')

# Aggregate payments by order_id
payments_agg = payments.groupby('order_id', as_index=False).agg({
    'payment_value': 'sum',
    'payment_installments': 'max',
    'payment_type': 'first'
})

print('Before:', payments.shape[0], '-> After:', payments_agg.shape[0])

Before: 103886 -> After: 99440


In [5]:
# Load required tables
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

# Aggregate items per order
items_agg = order_items.groupby('order_id', as_index=False).agg({
    'price': 'sum',
    'freight_value': 'sum',
    'product_id': 'count'
})
items_agg.rename(columns={'product_id': 'item_count'}, inplace=True)

# Merge products with English translation
products = products.merge(translation, on='product_category_name', how='left')

# The master merge — print shape after each step
df = orders.merge(customers, on='customer_id', how='left')
print('+ customers:', df.shape)
df = df.merge(payments_agg, on='order_id', how='left')
print('+ payments:', df.shape)
df = df.merge(items_agg, on='order_id', how='left')
print('+ items:', df.shape)

# Save the golden dataset
df = df.drop_duplicates()
df.to_csv('../data/processed/golden_dataset.csv', index=False)
print('Saved! Shape:', df.shape)

+ customers: (96470, 14)
+ payments: (96470, 17)
+ items: (96470, 20)
Saved! Shape: (96470, 20)
